# **ETL**

## Objectives

* view, cleanse and modify the dataset so its suitable for visualisation and ML downstream steps

## Inputs

* Medical Appointment No Shows from kaggle is the source (`https://www.kaggle.com/datasets/joniarroba/noshowappointments`)

## Outputs

* Cleansed file ready for usage, saved in data/processed


---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [30]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\ryan_\\VS-code-projects\\medical-appointment-no-show-analytics'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [31]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [32]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\ryan_\\VS-code-projects'

# Section 1 - pull data to local system

Setting up location for files on local system. 

In [33]:
data = 'data'
dwpath = os.path.join(current_dir, data, 'raw')
outpath = os.path.join(current_dir, data, 'processed')
print("Data Download Path:", dwpath)    
print("Data Cleansed Path:", outpath)

Data Download Path: c:\Users\ryan_\VS-code-projects\data\raw
Data Cleansed Path: c:\Users\ryan_\VS-code-projects\data\processed


> **Note:**  
> This step uses the Kaggle API and requires a valid `kaggle.json` credentials file configured on the local machine.  
> Without this file, the download step will fail on a new or clean environment.  
> If Kaggle is not configured, download the file manually from: [kaggle author page](https://www.kaggle.com/datasets/joniarroba/noshowappointments) 

In [34]:
# Import Kaggle API client class
from kaggle.api.kaggle_api_extended import KaggleApi

# Create API client instance
api = KaggleApi()

# Authenticate using local Kaggle credentials (kaggle.json)
api.authenticate()

# Download dataset ZIP from Kaggle into data/raw and unzip it there
api.dataset_download_files(
    "joniarroba/noshowappointments",
    path=dwpath,
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/joniarroba/noshowappointments


In [35]:
#checking the file downloaded is accessible and has records in it. 
import pandas as pd
df_raw = pd.read_csv(os.path.join(dwpath,'KaggleV2-May-2016.csv' ))

df_raw.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


---

# Section 2 - Data Inspection



## Initial Dataset Overview
Before performing any transformations, it is important to understand the structure of the dataset.

The following checks are performed:

* Dataset dimensions (rows and columns)
* Column names and feature descriptions
* Data types of each variable
* Summary statistics for numerical variables
This provides an initial understanding of the dataset and helps identify potential issues such as incorrect data types or unusual distributions.

## Preview the dataset & its shape

Displaying the first and last few rows helps confirm that the file loaded correctly and provides an initial view of the data structure.

In [36]:


print("Shape:", df_raw.shape)
display(df_raw.head(3)) # first 3 records
display(df_raw.sample(3)) # random 3 records
display(df_raw.tail(3)) # last 3 records


Shape: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
73445,3.465974e+14,5701830,F,2016-05-16T11:05:32Z,2016-05-16T00:00:00Z,28,MARIA ORTIZ,1,0,0,0,0,0,No
45036,3.265844e+12,5689129,F,2016-05-12T08:03:59Z,2016-05-24T00:00:00Z,53,SANTO ANTÔNIO,0,0,0,0,0,1,No
66665,5.237759e+14,5679036,F,2016-05-10T09:21:26Z,2016-05-10T00:00:00Z,55,JARDIM CAMBURI,0,0,0,0,0,0,No


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
110524,1.557663e+13,5630692,F,2016-04-27T16:03:52Z,2016-06-07T00:00:00Z,21,MARIA ORTIZ,0,0,0,0,0,1,No
110525,9.213493e+13,5630323,F,2016-04-27T15:09:23Z,2016-06-07T00:00:00Z,38,MARIA ORTIZ,0,0,0,0,0,1,No
110526,3.775115e+14,5629448,F,2016-04-27T13:30:56Z,2016-06-07T00:00:00Z,54,MARIA ORTIZ,0,0,0,0,0,1,No


## Review column names

This helps identify inconsistent naming, spelling issues, punctuation, or columns that may need renaming during the cleaning stage.

In [37]:
print("Column names:")
print(df_raw.columns.tolist())


Column names:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']


## Review data types

Inspecting data types helps identify whether columns are numeric, categorical, or date-like fields that may require conversion later.

In [38]:
# Display data types
print("Data types:")
print(df_raw.dtypes)

Data types:
PatientId         float64
AppointmentID       int64
Gender             object
ScheduledDay       object
AppointmentDay     object
Age                 int64
Neighbourhood      object
Scholarship         int64
Hipertension        int64
Diabetes            int64
Alcoholism          int64
Handcap             int64
SMS_received        int64
No-show            object
dtype: object


## Check for missing values

This step checks whether any columns contain null or missing values that may need to be handled during cleaning.

In [39]:
# Check missing values
missing_values = df_raw.isnull().sum()

print("Missing values per column:")
print(missing_values)

print("\nTotal missing values in dataset:", missing_values.sum())

Missing values per column:
PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

Total missing values in dataset: 0


## Check for duplicate rows

Duplicate rows can reduce data quality and may need to be removed if they do not represent genuine repeated records.

---

In [40]:
# Count duplicate rows
duplicate_count = df_raw.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


## Summary statistics

Summary statistics provide a quick overview of the numeric and categorical columns in the dataset.

In [41]:
# Summary statistics for numeric columns
print("Summary statistics for numeric columns:")
display(df_raw.describe())

# Summary statistics for categorical/object columns
print("Summary statistics for categorical columns:")
display(df_raw.describe(include="object"))

Summary statistics for numeric columns:


,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


Summary statistics for categorical columns:


,Gender,ScheduledDay,AppointmentDay,Neighbourhood,No-show
count,110527,110527,110527,110527,110527
unique,2,103549,27,81,2
top,F,2016-05-06T07:09:54Z,2016-06-06T00:00:00Z,JARDIM CAMBURI,No
freq,71840,24,4692,7717,88208


## Check unique values per column

This helps identify ID fields, binary fields, and high-cardinality columns such as neighbourhood names.

In [42]:
# Count unique values per column
print("Number of unique values per column:")
print(df_raw.nunique())

Number of unique values per column:
PatientId          62299
AppointmentID     110527
Gender                 2
ScheduledDay      103549
AppointmentDay        27
Age                  104
Neighbourhood         81
Scholarship            2
Hipertension           2
Diabetes               2
Alcoholism             2
Handcap                5
SMS_received           2
No-show                2
dtype: int64


## Inspect important business fields

Reviewing the target column and key categorical fields helps build an early understanding of the dataset before transformation.

In [43]:
# Inspect key columns
print("Value counts for 'No-show':")
print(df_raw["No-show"].value_counts(dropna=False))

print("\nValue counts for 'Gender':")
print(df_raw["Gender"].value_counts(dropna=False))

print("\nValue counts for 'Scholarship':")
print(df_raw["Scholarship"].value_counts(dropna=False))

print("\nValue counts for 'Hipertension':")
print(df_raw["Hipertension"].value_counts(dropna=False))

print("\nValue counts for 'SMS_received':")
print(df_raw["SMS_received"].value_counts(dropna=False))

Value counts for 'No-show':
No-show
No     88208
Yes    22319
Name: count, dtype: int64

Value counts for 'Gender':
Gender
F    71840
M    38687
Name: count, dtype: int64

Value counts for 'Scholarship':
Scholarship
0    99666
1    10861
Name: count, dtype: int64

Value counts for 'Hipertension':
Hipertension
0    88726
1    21801
Name: count, dtype: int64

Value counts for 'SMS_received':
SMS_received
0    75045
1    35482
Name: count, dtype: int64


## Check for obvious anomalies

A quick anomaly review helps identify clearly invalid values, such as negative ages, before moving into formal cleaning and feature engineering.

In [44]:
# Check for negative ages
negative_age_count = (df_raw["Age"] < 0).sum()
print("Rows with negative age values:", negative_age_count)

# Check age range
print("Minimum age:", df_raw["Age"].min())
print("Maximum age:", df_raw["Age"].max())

# Preview invalid age rows if any exist
if negative_age_count > 0:
    print("\nRows with negative age values:")
    display(df_raw[df_raw["Age"] < 0].head())

Rows with negative age values: 1
Minimum age: -1
Maximum age: 115

Rows with negative age values:


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No


## Inspect raw date columns

The dataset contains both a booking date and an appointment date. These will later be converted to datetime format so that lead time can be calculated.

In [45]:
# Preview raw date columns
print("values from ScheduledDay:")
display(df_raw["ScheduledDay"].head())

print("values from AppointmentDay:")
display(df_raw["AppointmentDay"].head())

values from ScheduledDay:


0    2016-04-29T18:38:08Z
1    2016-04-29T16:08:27Z
2    2016-04-29T16:19:04Z
3    2016-04-29T17:29:31Z
4    2016-04-29T16:07:23Z
Name: ScheduledDay, dtype: object

values from AppointmentDay:


0    2016-04-29T00:00:00Z
1    2016-04-29T00:00:00Z
2    2016-04-29T00:00:00Z
3    2016-04-29T00:00:00Z
4    2016-04-29T00:00:00Z
Name: AppointmentDay, dtype: object

## Test datetime conversion and calculate temporary wait time

This step checks whether the date fields convert correctly and whether there are any impossible appointment timings, such as negative wait days.

In [46]:
# Convert raw date columns to datetime for inspection
scheduled_temp = pd.to_datetime(df_raw["ScheduledDay"], errors="coerce", utc=True)
appointment_temp = pd.to_datetime(df_raw["AppointmentDay"], errors="coerce", utc=True)

# Check whether any values failed conversion
print("Invalid ScheduledDay values after conversion:", scheduled_temp.isna().sum())
print("Invalid AppointmentDay values after conversion:", appointment_temp.isna().sum())

# Create temporary wait-days calculation using date only
wait_days_temp = (
    appointment_temp.dt.normalize() - scheduled_temp.dt.normalize()
).dt.days

print("\nTemporary wait_days summary:")
display(wait_days_temp.describe())

print("Rows where wait_days is negative:", (wait_days_temp < 0).sum())

Invalid ScheduledDay values after conversion: 0
Invalid AppointmentDay values after conversion: 0

Temporary wait_days summary:


count    110527.000000
mean         10.183702
std          15.254996
min          -6.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
dtype: float64

Rows where wait_days is negative: 5


## ETL observations from raw data inspection

The raw dataset was successfully loaded and reviewed to understand its structure before cleaning.

Initial inspection focused on:
- dataset dimensions
- column names
- data types
- missing values
- duplicate rows
- early anomaly checks

This step supports later ETL decisions such as:
- renaming inconsistent columns
- converting date fields to datetime format
- checking invalid ages
- creating derived features such as wait time between booking and appointment
- validating whether any records should be removed due to impossible date relationships

These checks improve transparency and provide evidence of a structured ETL process before analysis and dashboard development.

## Section 3 - Data Cleaning and Feature Engineering

## Rename columns and convert dates

Column names are standardised and the date fields are converted into datetime format.

This makes the dataset easier to work with and prepares it for feature engineering.

In [47]:
# Create a working copy of the raw dataset
df = df_raw.copy()

# Rename columns for consistency and readability
df = df.rename(columns={
    "PatientId": "patient_id",
    "AppointmentID": "appointment_id",
    "ScheduledDay": "scheduled_day",
    "AppointmentDay": "appointment_day",
    "Hipertension": "hypertension",
    "Handcap": "handicap",
    "SMS_received": "sms_received",
    "No-show": "no_show",
    "Age": "age"
})

# Preview updated column names
print(df.columns.tolist())

['patient_id', 'appointment_id', 'Gender', 'scheduled_day', 'appointment_day', 'age', 'Neighbourhood', 'Scholarship', 'hypertension', 'Diabetes', 'Alcoholism', 'handicap', 'sms_received', 'no_show']


## Convert date columns

The booking and appointment fields are converted to datetime format so they can be used in time-based analysis.

In [48]:
# Convert date columns to datetime
df["scheduled_day"] = pd.to_datetime(df["scheduled_day"], errors="coerce", utc=True)
df["appointment_day"] = pd.to_datetime(df["appointment_day"], errors="coerce", utc=True)

# Check conversion worked
print(df[["scheduled_day", "appointment_day"]].dtypes)

scheduled_day      datetime64[ns, UTC]
appointment_day    datetime64[ns, UTC]
dtype: object


## Create date-based features

New columns are created to support analysis of appointment timing and patient no-show behaviour.

In [49]:
# Create date-only fields
df["scheduled_date"] = df["scheduled_day"].dt.normalize()
df["appointment_date"] = df["appointment_day"].dt.normalize()

# Create wait time in days
df["wait_days"] = (df["appointment_date"] - df["scheduled_date"]).dt.days

# Create useful time-based fields
df["scheduled_hour"] = df["scheduled_day"].dt.hour
df["scheduled_day_of_week"] = df["scheduled_day"].dt.day_name()
df["appointment_day_of_week"] = df["appointment_day"].dt.day_name()

# Preview selected columns
display(df[[
    "scheduled_day",
    "appointment_day",
    "scheduled_date",
    "appointment_date",
    "wait_days",
    "scheduled_hour",
    "scheduled_day_of_week",
    "appointment_day_of_week"
]].head())

,scheduled_day,appointment_day,scheduled_date,appointment_date,wait_days,scheduled_hour,scheduled_day_of_week,appointment_day_of_week
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,0,18,Friday,Friday
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,0,16,Friday,Friday
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,0,16,Friday,Friday
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,0,17,Friday,Friday
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,2016-04-29 00:00:00+00:00,0,16,Friday,Friday


## Create simple analytical fields

A few extra fields are added to make later analysis and visualisation easier.

In [50]:
# Convert target to binary flag
df["no_show_flag"] = df["no_show"].map({"No": 0, "Yes": 1})

# Simplify handicap into a binary field
df["has_handicap"] = (df["handicap"] > 0).astype(int)

# Flag same-day appointments
df["same_day_appointment"] = (df["wait_days"] == 0).astype(int)

# Preview new fields
display(df[[
    "no_show",
    "no_show_flag",
    "handicap",
    "has_handicap",
    "wait_days",
    "same_day_appointment"
]].head())

,no_show,no_show_flag,handicap,has_handicap,wait_days,same_day_appointment
0,No,0,0,0,0,1
1,No,0,0,0,0,1
2,No,0,0,0,0,1
3,No,0,0,0,0,1
4,No,0,0,0,0,1


Remove records with negative ages, as these are invalid.
Age values of 0 are retained because they may represent infants under 1 year old.

In [51]:
# Check row count before cleaning
print("Rows before cleaning:", df.shape[0])

# Remove negative ages only
# age = 0 is retained because it may represent babies under 1 year old
df = df[df["age"] >= 0]

# Remove impossible negative wait times
df = df[df["wait_days"] >= 0]

# Check row count after cleaning
print("Rows after cleaning:", df.shape[0])

Rows before cleaning: 110527
Rows after cleaning: 110521


Identifier fields (patient_id and appointment_id) were removed from the analysis dataset because they do not add explanatory value to the business questions and retaining them would not support privacy-aware, governance-focused analysis.

In [52]:
# Drop identifier columns before saving the final cleaned dataset
df = df.drop(columns=["patient_id", "appointment_id"])


## Save cleaned dataset

The cleaned dataset is saved for use in the EDA and modelling notebooks.

In [53]:
# Ensure the processed data folder exists
os.makedirs(outpath, exist_ok=True)

# Define output file path
output_file = os.path.join(outpath, "medical_appointments_cleaned.csv")

# Add row_id as a unique identifier for each record for possible future use in visualization section for ethics review purposes, we will not use patient_id or appointment_id as they could potentially be re-identifiable.
df["row_id"] = df.index

# Save cleaned dataset
df.to_csv(output_file, index=False)

print("Cleaned dataset saved to:", output_file)
print("Saved dataset shape:", df.shape)

Cleaned dataset saved to: c:\Users\ryan_\VS-code-projects\data\processed\medical_appointments_cleaned.csv
Saved dataset shape: (110521, 22)


## Summary

In this step, column names were standardised, date fields were converted, new time-based features were created, and invalid rows & non required columns were removed.

The cleaned dataset was then saved for the next stage of the project.

---